# 04 — Residual Encryption with AEAD

After snapping to a 250 m tile, the residual `(rx, ry)` contains the full GPS precision within the tile — up to ±125 m in each axis. If stored in plaintext, an attacker with the tile index (even encrypted) could reconstruct the exact location once they obtain the PRP key.

**AEAD (Authenticated Encryption with Associated Data)** gives two guarantees:

1. **Confidentiality** — nobody without the key can read `(rx, ry)`.
2. **Integrity** — any modification to the ciphertext, nonce, or associated data is detected and decryption returns `None`.

We use **ChaCha20-Poly1305**, standardised in TLS 1.3 (RFC 7539). If the `cryptography` package is absent, the library falls back to XOR+HMAC-SHA256 which provides the same security properties.

The nonce must be **unique per encryption**; reuse with the same key would allow an attacker to XOR two ciphertexts and cancel the keystream entirely. The `encode()` method generates `secrets.token_bytes(12)` for every call.

<div style="background:#f5faf9;border:1px solid #b8ddd8;border-radius:8px;padding:12px 14px 14px;margin:10px 0 22px;font-family:sans-serif;">
<div style="font-size:11px;color:#5a9e99;margin-bottom:10px;font-style:italic;">Pipeline step 3 of 4 — AEAD encryption of the sub-tile residual</div>
<div style="display:flex;align-items:stretch;">
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:0;position:relative;z-index:4;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB02</div><div style="font-weight:500;font-size:13px;">① Project</div></div>
    <div style="background:#dff0ee;color:#3d7a71;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:3;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB03</div><div style="font-weight:500;font-size:13px;">② Snap+Shuffle</div></div>
    <div style="background:#2a9d8f;color:white;clip-path:polygon(0 0,calc(100% - 22px) 0,100% 50%,calc(100% - 22px) 100%,0 100%);padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:2;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB04</div><div style="font-weight:700;font-size:13px;">③ Lock</div></div>
    <div style="background:#dff0ee;color:#3d7a71;padding:10px 18px 10px 18px;margin-left:-21px;position:relative;z-index:1;min-width:130px;text-align:center;"><div style="font-size:10px;opacity:0.85;margin-bottom:3px;">NB05</div><div style="font-weight:500;font-size:13px;">④ Wobble</div></div>
</div>
</div>

## Learning Objectives

By the end of this notebook you will be able to:

1. **Identify** the three components of ChaCha20-Poly1305 — the stream cipher, the Poly1305 authenticator, and the associated data field.
2. **Explain** what semantic security guarantees and why a fresh nonce per record is essential for AEAD correctness.
3. **Construct** the length-prefixed associated data string for a given `(qx, qy, tweak)` tuple and verify it prevents boundary-shift attacks.
4. **Analyze** the six tamper scenarios and classify each by the specific authentication failure mode it exercises.
5. **Appraise** which residual threats remain outside the AEAD threat model and require separate mitigations.

In [1]:
import struct
import hashlib
import secrets
import numpy as np
import plotly.express as px

from map_encryption import _AEAD, _build_ad, _CHACHA_AVAILABLE, _project, SchemeParams

print(f'AEAD backend: {"ChaCha20-Poly1305 (cryptography)" if _CHACHA_AVAILABLE else "XOR+HMAC-SHA256 (fallback)"}')

params = SchemeParams()
BIN = params.bin_size_m
CENTER_LAT, CENTER_LON = 51.513341, -0.136668  # Broadwick Street pump, Soho, London (1854 cholera outbreak)

AEAD backend: ChaCha20-Poly1305 (cryptography)


In [2]:
aead = _AEAD(secrets.token_bytes(32))
pt = struct.pack('>dd', 42.5, -8.3)  # a residual (rx, ry)

nonces = [secrets.token_bytes(12) for _ in range(3)]
ciphertexts = [aead.encrypt(n, pt, b'') for n in nonces]

print('3 encryptions of the same plaintext (rx=42.5, ry=-8.3):')
for i, ct in enumerate(ciphertexts):
    print(f'  ct[{i}] = {ct.hex()[:32]}...')

# Verify all 3 are distinct
assert len(set(ct.hex() for ct in ciphertexts)) == 3, 'Ciphertexts should be distinct'

# Verify all 3 decrypt correctly
for i, (n, ct) in enumerate(zip(nonces, ciphertexts)):
    recovered = aead.decrypt(n, ct, b'')
    assert recovered is not None, f'Decryption {i} returned None unexpectedly'
    rx_r, ry_r = struct.unpack('>dd', recovered)
    assert abs(rx_r - 42.5) < 1e-12 and abs(ry_r + 8.3) < 1e-12

print('All 3 decrypt correctly.')
print('3 encryptions of the same plaintext -> 3 completely different ciphertexts')

3 encryptions of the same plaintext (rx=42.5, ry=-8.3):
  ct[0] = 38d990160ace1a31ce4e9fffda18dbd3...
  ct[1] = 411e68908d6a25ceb03dc4fe7123108d...
  ct[2] = 90d0f408a6bcef144ef59199fe324e08...
All 3 decrypt correctly.
3 encryptions of the same plaintext -> 3 completely different ciphertexts


In [3]:
import pandas as pd

rows = []
for cid, ct in enumerate(ciphertexts):
    for bidx, bval in enumerate(ct):
        rows.append({'byte_index': bidx, 'value': bval,
                     'ciphertext_id': f'ct[{cid}]'})

df_bytes = pd.DataFrame(rows)
fig = px.scatter(
    df_bytes, x='byte_index', y='value', color='ciphertext_id',
    title='Same plaintext, 3 different nonces: byte-level comparison',
    labels={'byte_index': 'Byte index', 'value': 'Byte value (0-255)'},
    template='plotly_white',
    height=400,
    opacity=0.7
)
fig.show()

**Figure 4a.** Table of six tamper-detection scenarios applied to a Broadwick Street pump record — flipped ciphertext bit, changed nonce, wrong associated data, wrong AEAD key, and two baseline checks — showing whether ChaCha20-Poly1305 tag verification succeeds or returns None.

## Associated Data Construction

`_build_ad(qx, qy, tweak)` packs `(qx, qy, len(tweak))` as big-endian integers then appends the raw tweak bytes:

```python
struct.pack('>iiI', qx, qy, len(tweak)) + tweak
```

The AD is **authenticated but not encrypted**. If an attacker moves a ciphertext from tile A to tile B, decryption reconstructs a different AD (different qx/qy) and the Poly1305 tag verification fails — the record is rejected.

**Why length-prefix the tweak?** Without it, two AD constructions can collide:

```
qx=100, qy=23,  tweak=b'6'    →  pack(100,23) + b'6'
qx=100, qy=236, tweak=b''     →  pack(100,236) + b''
```

The raw integer encodings of `qy=23` and `qy=236` are different (4 bytes each), so the collision actually requires a more subtle shift. With the length prefix for the tweak, the concatenated bytes are unambiguous regardless of content.

In [4]:
ad1 = _build_ad(100, 23, b'6')
ad2 = _build_ad(100, 236, b'')
print(f'_build_ad(100, 23,  b"6") = {ad1.hex()}')
print(f'_build_ad(100, 236, b"")  = {ad2.hex()}')
assert ad1 != ad2, 'AD values should differ'
print('AD values differ: OK')

# Demonstrate: encrypt with ad1, try to decrypt with ad2
demo_aead = _AEAD(secrets.token_bytes(32))
demo_nonce = secrets.token_bytes(12)
demo_pt = struct.pack('>dd', 10.0, 20.0)
demo_ct = demo_aead.encrypt(demo_nonce, demo_pt, ad1)
result = demo_aead.decrypt(demo_nonce, demo_ct, ad2)
assert result is None, 'Expected None for wrong AD'
print('Decryption with mismatched AD returned None: OK')
print('Boundary-shift attack correctly blocked.')

_build_ad(100, 23,  b"6") = 00000064000000170000000136
_build_ad(100, 236, b"")  = 00000064000000ec00000000
AD values differ: OK
Decryption with mismatched AD returned None: OK
Boundary-shift attack correctly blocked.


In [5]:
# Build a realistic encode scenario
test_key = secrets.token_bytes(32)
aead2 = _AEAD(test_key)
x_ts, y_ts = _project(CENTER_LAT, CENTER_LON)
qx_ts = int(round(x_ts / BIN))
qy_ts = int(round(y_ts / BIN))
rx_ts = x_ts - qx_ts * BIN
ry_ts = y_ts - qy_ts * BIN
nonce_ts = secrets.token_bytes(12)
ad_ts = _build_ad(qx_ts, qy_ts, b'nb04-demo')
pt_ts = struct.pack('>dd', rx_ts, ry_ts)
ct_ts = aead2.encrypt(nonce_ts, pt_ts, ad_ts)

scenarios = [
    ('Flip ct byte',      lambda: aead2.decrypt(nonce_ts, bytes([ct_ts[0]^1]) + ct_ts[1:], ad_ts)),
    ('Change nonce',      lambda: aead2.decrypt(secrets.token_bytes(12), ct_ts, ad_ts)),
    ('Change qx in AD',   lambda: aead2.decrypt(nonce_ts, ct_ts, _build_ad(qx_ts+1, qy_ts, b'nb04-demo'))),
    ('Change qy in AD',   lambda: aead2.decrypt(nonce_ts, ct_ts, _build_ad(qx_ts, qy_ts+1, b'nb04-demo'))),
    ('Change tweak in AD',lambda: aead2.decrypt(nonce_ts, ct_ts, _build_ad(qx_ts, qy_ts, b'wrong-tweak'))),
    ('Wrong key',         lambda: _AEAD(secrets.token_bytes(32)).decrypt(nonce_ts, ct_ts, ad_ts)),
]

print(f'{"Scenario":<25} {"Result"}')
print('-' * 40)
all_none = True
for name, fn in scenarios:
    result = fn()
    status = 'PASS (None)' if result is None else 'FAIL (got data!)'
    if result is not None:
        all_none = False
    print(f'  {name:<23} {status}')

assert all_none, 'At least one tamper scenario was not rejected'
print('\nAll 6 tamper scenarios correctly rejected.')

Scenario                  Result
----------------------------------------
  Flip ct byte            PASS (None)
  Change nonce            PASS (None)
  Change qx in AD         PASS (None)
  Change qy in AD         PASS (None)
  Change tweak in AD      PASS (None)
  Wrong key               PASS (None)

All 6 tamper scenarios correctly rejected.


## AEAD Tamper-Detection Results

| Scenario | What was modified | Outcome |
|----------|-------------------|---------|
| Baseline (correct) | (nothing) | ACCEPTED |
| Flip ct byte | First byte of ciphertext | REJECTED |
| Change nonce | Nonce (12-byte random value) | REJECTED |
| Change qx in AD | qx in Associated Data | REJECTED |
| Change qy in AD | qy in Associated Data | REJECTED |
| Change tweak in AD | Tweak in Associated Data | REJECTED |
| Wrong key | Encryption key | REJECTED |

Every modification causes Poly1305 tag verification to fail and `decrypt()` returns `None`.

## What AEAD Does NOT Protect

1. **Does not hide that an encryption happened.** The presence of `ct_resid` in a database is visible to anyone with database read access.

2. **Does not protect `qxp`, `qyp`.** Tile index confidentiality is the PRP's job. AEAD only protects the sub-tile residual.

3. **Nonce reuse is catastrophic.** If the same nonce is ever used twice with the same AEAD key, an attacker can XOR the two ciphertexts: `ct1 XOR ct2 = pt1 XOR pt2`, cancelling the keystream and revealing the XOR of the two plaintexts. The `encode()` method generates `secrets.token_bytes(12)` on every call — 96-bit random nonces — making collision probability negligible.

## References

- **Snow, J.** (1855). *On the Mode of Communication of Cholera* (2nd ed.). Churchill, London. — Source of the 1854 Soho cholera death and pump location dataset used throughout these notebooks.
- **Nir, Y., & Langley, A.** (2018). ChaCha20 and Poly1305 for IETF Protocols. RFC 8439. IETF. — Specification for the AEAD cipher used in this library.
- **Aumasson, J.-P., Neves, S., Wilcox-O'Halon, Z., & Winnerlein, C.** (2013). BLAKE2: simpler, smaller, fast as MD5. *ACNS 2013*, LNCS 7954, 119–135. — BLAKE2s is the PRF used for key derivation, the Feistel round function, and display jitter.

## Glossary

| Term | Definition |
|------|-----------|
| **AEAD** | Authenticated Encryption with Associated Data; simultaneously encrypts a plaintext and produces an authentication tag that covers both the ciphertext and unencrypted associated data. |
| **ChaCha20-Poly1305** | The AEAD cipher used in this scheme; ChaCha20 is a stream cipher providing confidentiality, Poly1305 is a MAC providing integrity and authenticity. |
| **Nonce** | A 12-byte value that must be unique per encryption; used to initialise ChaCha20 and included unencrypted in the record. |
| **Plaintext** | The unencrypted residual `(rx, ry)` packed as two 64-bit floats (16 bytes). |
| **Ciphertext (`ct_resid`)** | The encrypted output: 16 bytes of encrypted plaintext plus a 16-byte Poly1305 authentication tag (32 bytes total). |
| **Associated Data (AD)** | Unencrypted context data authenticated by the AEAD tag: `(qx, qy, tweak)`; binding the ciphertext to the specific tile and purpose. |
| **Authentication tag** | The 16-byte Poly1305 output appended to `ct_resid`; any modification to the ciphertext, nonce, or AD causes tag verification to fail. |
| **IND-CPA** | Indistinguishability under Chosen-Plaintext Attack; the security property ensuring that encryptions of different plaintexts are computationally indistinguishable. |
| **Semantic security** | The property that the ciphertext reveals no partial information about the plaintext; implied by IND-CPA. |
| **Tamper detection** | The ability to detect any modification to `ct_resid`, `nonce`, or AD; provided by the Poly1305 tag with overwhelming probability. |